**Features** — history of the same starting offset, available at the time of the write.

| Column | Feature before transformation |
|---|---|
| 0 | Write offset, bytes |
| 1 | Write size, bytes |
| 2 | Number of previous reads at this offset |
| 3 | Number of previous writes at this offset |
| 4 | Seconds since the previous read at this offset |
| 5 | Seconds since the previous write at this offset |
| 6 | Most recent completed interval between reads at this offset, seconds |

Each feature is transformed with `log1p`. Missing history is represented by `NaN`. usr_1 uses all seven columns; prxy_1 uses `[0, 1, 4, 5, 6]`.

**Label** — seconds from a write to the first subsequent read with the same starting offset. Intervening writes do not invalidate the sample. Samples without a read before the end of their time window are excluded from training/evaluation. The model is trained on `log1p(label)`; predictions in seconds are obtained as `max(0, expm1(prediction))`.


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np

model_directory = Path('.')
if Path('submission/usr_1.lightgbm.txt').is_file():
    model_directory = Path('submission')

models = {}
for workload in ('usr_1', 'prxy_1'):
    model_path = model_directory / f'{workload}.lightgbm.txt'
    models[workload] = lgb.Booster(model_file=str(model_path))

In [2]:
{workload: model.feature_name() for workload, model in models.items()}

{'usr_1': ['log_offset_bytes',
  'log_size_bytes',
  'log_prior_read_count',
  'log_prior_write_count',
  'log_seconds_since_read',
  'log_seconds_since_write',
  'log_last_read_read_interval_seconds'],
 'prxy_1': ['log_offset_bytes',
  'log_size_bytes',
  'log_seconds_since_read',
  'log_seconds_since_write',
  'log_last_read_read_interval_seconds']}

In [3]:
FEATURE_COLUMNS = {
    'usr_1': [0, 1, 2, 3, 4, 5, 6],
    'prxy_1': [0, 1, 4, 5, 6],
}


def predict_seconds(workload, raw_features):
    raw_features = np.atleast_2d(np.asarray(raw_features, dtype=float))
    log_features = np.log1p(raw_features)
    selected_features = log_features[:, FEATURE_COLUMNS[workload]]

    log_predictions = models[workload].predict(selected_features, num_threads=2)
    return np.maximum(0, np.expm1(log_predictions))

In [4]:
recorded_results = [
    {
        'workload': 'usr_1',
        'day': 2,
        'observed_intervals': 66_654,
        'coverage_fraction': 0.1783828484412175,
        'baseline_mae_seconds': 9918.773740623254,
        'model_mae_seconds': 12153.362087755688,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'usr_1',
        'day': 3,
        'observed_intervals': 3_577,
        'coverage_fraction': 0.02094102907858301,
        'baseline_mae_seconds': 13291.457813713294,
        'model_mae_seconds': 15071.470434888139,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'usr_1',
        'day': 4,
        'observed_intervals': 215_104,
        'coverage_fraction': 0.34996461372011484,
        'baseline_mae_seconds': 16450.66607042449,
        'model_mae_seconds': 24658.94281783792,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'prxy_1',
        'day': 2,
        'observed_intervals': 4_451_738,
        'coverage_fraction': 0.5392348543550198,
        'baseline_mae_seconds': 10.04810576824427,
        'model_mae_seconds': 5.559757821067748,
        'evaluation_role': 'previously_evaluated_replication',
    },
    {
        'workload': 'prxy_1',
        'day': 3,
        'observed_intervals': 4_313_417,
        'coverage_fraction': 0.5369413682602354,
        'baseline_mae_seconds': 11.015346296963763,
        'model_mae_seconds': 7.926027052344753,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'prxy_1',
        'day': 4,
        'observed_intervals': 4_270_777,
        'coverage_fraction': 0.5371556006332,
        'baseline_mae_seconds': 13.665613102591298,
        'model_mae_seconds': 9.647575508444456,
        'evaluation_role': 'new_future_period',
    },
]

In [5]:
header = (
    f'{"trace":<8} {"day":>3} {"n":>9} {"coverage_%":>11} '
    f'{"baseline_mae_s":>15} {"model_mae_s":>12}  {"evaluation_role"}'
)
print(header)

for result in recorded_results:
    coverage_percent = 100 * result['coverage_fraction']
    print(
        f'{result["workload"]:<8} {result["day"]:>3} '
        f'{result["observed_intervals"]:>9,} {coverage_percent:>11.2f} '
        f'{result["baseline_mae_seconds"]:>15.2f} '
        f'{result["model_mae_seconds"]:>12.2f}  '
        f'{result["evaluation_role"]}'
    )

trace    day         n  coverage_%  baseline_mae_s  model_mae_s  evaluation_role
usr_1      2    66,654       17.84         9918.77     12153.36  new_future_period
usr_1      3     3,577        2.09        13291.46     15071.47  new_future_period
usr_1      4   215,104       35.00        16450.67     24658.94  new_future_period
prxy_1     2 4,451,738       53.92           10.05         5.56  previously_evaluated_replication
prxy_1     3 4,313,417       53.69           11.02         7.93  new_future_period
prxy_1     4 4,270,777       53.72           13.67         9.65  new_future_period
